In [12]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import warnings
from scipy.stats import kruskal
import re

warnings.filterwarnings('ignore')

#plt.rcParams['font.family'] = 'AppleGothic'    # Mac
plt.rcParams['font.family'] = 'Malgun Gothic' # Windows
plt.rcParams['axes.unicode_minus'] = False

print("라이브러리 로드 완료")

라이브러리 로드 완료


In [13]:
reviews = pd.read_csv("../../../../data/preprocessed/steam_indie_reviews.csv")
df = pd.read_csv('../../../../data/preprocessed/steam_indie_genre_stratified_sample.csv')

In [ ]:
genre_info = df[["appid", "primary_genre"]].drop_duplicates()

reviews_merged = reviews.merge(
    genre_info,
    on="appid",
    how="left"
)
reviews = reviews_merged

reviews["sentiment"] = reviews["voted_up"].map({
    True: "positive",
    False: "negative"
})

reviews[["appid", "language", "review", "voted_up", "sentiment"]].head()

reviews_en = reviews[
    (reviews["language"] == "english") &
    (reviews["review"].notna())
].copy()

reviews_en.shape




from sklearn.feature_extraction.text import TfidfVectorizer
def get_tfidf_bigrams(data, sentiment_value, n=100):
    texts = data[data["sentiment"] == sentiment_value]["clean_review"]

    tfidf = TfidfVectorizer(
        stop_words="english",
        min_df=3,
        max_df=0.7,
        ngram_range=(2, 2)
    )

    X = tfidf.fit_transform(texts)

    result = pd.DataFrame({
        "term": tfidf.get_feature_names_out(),
        "score": X.sum(axis=0).A1
    }).sort_values("score", ascending=False)

    return result.head(n)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

reviews_en["clean_review"] = reviews_en["review"].apply(clean_text)

reviews_en[["review", "clean_review"]].head()

positive_bigram_tfidf = get_tfidf_bigrams(reviews_en, "positive", 300)
negative_bigram_tfidf = get_tfidf_bigrams(reviews_en, "negative", 300)

noise_bigrams = [
    "good game", "fun game", "great game", "game fun", "game great",
    "game play", "love game", "game good", "recommend game",
    "game really", "play game", "like game", "game just",
    "game like", "really fun", "really good", "little game",
    "game love", "games like", "game ve", "playing game",
    "pretty good", "game lot", "buy game", "game amazing",
    "best game", "game feels", "game game", "game worth",
    "played game", "felt like", "feel like", "feels like",
    "game recommend", "amazing game", "ve played",
    "fun play", "nice game", "cool game",

    # 추가 제거
    "highly recommend", "super fun", "lot fun", "cute game",
    "puzzle game", "horror game", "parkour game", "golf game",
    "game year", "game played", "games ve", "praying great"
]

game_title_terms = [
    "duck detective",
    "luck landlord",
    "cod zombies"
]

game_character = [
    "great mita"
]

remove_terms = set(noise_bigrams + game_title_terms + game_character)

positive_bigram_clean = positive_bigram_tfidf[
    ~positive_bigram_tfidf["term"].isin(remove_terms)
].copy()

positive_bigram_clean = positive_bigram_tfidf[
    ~positive_bigram_tfidf["term"].str.contains(r"\bgame\b", regex=True)
].copy()


positive_bigram_clean = positive_bigram_clean[
    ~positive_bigram_clean["term"].isin(remove_terms)
].copy()

issue_keywords = {
    "버그/크래시": [
        "bug", "bugs", "crash", "crashes", "crashed",
        "glitch", "broken", "freeze", "freezes"
    ],
    "최적화/성능": [
        "lag", "fps", "performance", "optimization",
        "stutter", "slow", "frame"
    ],
    "콘텐츠 부족": [
        "lack content", "no content", "short", "empty",
        "repetitive", "boring", "grind"
    ],
    "가격/가성비": [
        "price", "expensive", "refund", "money",
        "not worth", "worth", "value"
    ],
    "조작/UX": [
        "control", "controls", "ui", "menu",
        "tutorial", "interface"
    ],
    "난이도/밸런스": [
        "difficulty", "hard", "unfair", "balance",
        "too easy", "too hard"
    ],
    "멀티/서버": [
        "server", "servers", "disconnect",
        "online", "matchmaking", "multiplayer",
        "friends", "co op", "together"
    ],
    "아트/비주얼": [
        "art", "art style", "pixel art", "visual",
        "graphics", "beautiful", "animation"
    ],
    "스토리/캐릭터": [
        "story", "character", "characters",
        "dialogue", "writing", "voice acting"
    ],
    "음악/사운드": [
        "music", "soundtrack", "sound", "audio"
    ],
    "게임플레이": [
        "gameplay", "combat", "mechanic", "mechanics",
        "level design", "puzzle"
    ],
    "분위기/감성": [
        "atmosphere", "cozy", "relaxing",
        "charming", "cute"
    ],
    "반복 플레이/중독성": [
        "addictive", "replay", "replayable", "loop"
    ]
}

def classify_review_issues(text, issue_keywords):
    if pd.isna(text):
        return ["기타"]

    text = str(text).lower()
    matched_categories = []

    for category, keywords in issue_keywords.items():
        for keyword in keywords:
            pattern = r"\b" + re.escape(keyword.lower()) + r"\b"

            if re.search(pattern, text):
                matched_categories.append(category)
                break

    if len(matched_categories) == 0:
        return ["기타"]

    return matched_categories


reviews_en["issue_categories"] = reviews_en["clean_review"].apply(
    lambda x: classify_review_issues(x, issue_keywords)
)

reviews_en[["clean_review", "issue_categories"]].head(20)


issue_count = (
    reviews_en.explode("issue_categories")
      .groupby("issue_categories")
      .size()
      .reset_index(name="review_count")
      .sort_values("review_count", ascending=False)
)

issue_count["ratio"] = issue_count["review_count"] / len(reviews_en) * 100

issue_by_sentiment = (
    reviews_en.explode("issue_categories")
      .groupby(["sentiment", "issue_categories"])
      .size()
      .reset_index(name="review_count")
      .sort_values(["sentiment", "review_count"], ascending=[True, False])
)

issue_by_genre = (
    reviews_en.explode("issue_categories")
      .groupby(["primary_genre", "issue_categories","sentiment"])
      .size()
      .reset_index(name="review_count")
      .sort_values(["primary_genre", "review_count"], ascending=[True, False])
)

positive = issue_by_genre[issue_by_genre['sentiment']=='positive']

negative = issue_by_genre[issue_by_genre['sentiment']=='negative']

In [15]:
positive

,primary_genre,issue_categories,sentiment,review_count
5,Action,기타,positive,6251
3,Action,게임플레이,positive,1769
17,Action,스토리/캐릭터,positive,1013
21,Action,음악/사운드,positive,986
1,Action,가격/가성비,positive,954
...,...,...,...,...
209,Strategy,버그/크래시,positive,266
207,Strategy,반복 플레이/중독성,positive,221
217,Strategy,음악/사운드,positive,187
221,Strategy,최적화/성능,positive,174
